# 7주차 ① 합성곱 연산의 원리 — 실습 1~3

**목표**: 커널을 직접 바꿔 가며 *"커널 = 찾고 싶은 무늬"* 를 눈으로 확인하고,
**출력 크기를 손으로 계산해** `Conv2d` 실제 출력과 대조하며,
층별 특징맵을 보고 **층이 깊어질수록 보는 것이 달라진다**를 관찰한다.

> **실행 전 확인** — 커널 `Python (dl2026)`.
> ⚠️ **CIFAR-10 다운로드가 약 170MB 입니다.** 셀 1을 **지금 바로** 실행해 두세요.

### 오늘 새로 배우는 건 `nn.Conv2d` 한 줄과 그 원리뿐입니다

학습 루프도, 옵티마이저도, 평가도 **6주차 그대로**입니다.

```
   [원래 이미지 28×28]                   [Flatten 후 784개]

     ■ ■ ■ □ □                            ■ ■ ■ □ □ ■ □ □ ■ ■ ...
     ■ □ ■ □ □          →  펴기  →        ↑     ↑
     ■ ■ ■ □ □                            이 둘이 "위아래로 붙어 있었다"는
                                          정보가 완전히 사라진다
```

```
   CIFAR-10 : 32 × 32 × 3(컬러) = 3,072 입력

     MLP  : 3072 × 512 = 1,572,864 개        ← 첫 층만으로 157만
     CNN  : 3×3 커널 × 3채널 × 32개 = 864 개  ← 첫 층 파라미터
                                    약 1,800배 차이
```

> **핵심 메시지 ★★ (출제 1순위)**: MLP 가 이미지에 불리한 이유는 **두 가지** —
> **① 공간 정보 소실**, **② 파라미터 폭증**. 이 두 문장을 그대로 답할 수 있어야 합니다.

## 실습 1 — 커널을 손으로 적용

In [ ]:
# 셀 1 — 이미지 한 장 준비  ★ 다운로드가 시작되면 그대로 두세요 (약 170MB)
import torch, torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

ROOT = "data"
train = datasets.CIFAR10(ROOT, train=True, download=True, transform=transforms.ToTensor())
CLASSES = ["비행기","자동차","새","고양이","사슴","개","개구리","말","배","트럭"]

img, label = train[7]
gray = img.mean(dim=0, keepdim=True).unsqueeze(0)      # (1,1,32,32) 흑백으로
print("입력 shape :", gray.shape, "| 클래스 :", CLASSES[label])
plt.imshow(gray[0,0], cmap="gray"); plt.axis("off"); plt.show()

In [ ]:
# 셀 2 — 커널 3종을 직접 적용
kernels = {
    "세로 경계": [[ 1, 0,-1], [ 1, 0,-1], [ 1, 0,-1]],
    "가로 경계": [[ 1, 1, 1], [ 0, 0, 0], [-1,-1,-1]],
    "흐리게":    [[1/9]*3, [1/9]*3, [1/9]*3],
}

fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
ax[0].imshow(gray[0,0], cmap="gray"); ax[0].set_title("원본"); ax[0].axis("off")

for i, (name, k) in enumerate(kernels.items(), 1):
    w = torch.tensor(k, dtype=torch.float32).reshape(1, 1, 3, 3)
    out = F.conv2d(gray, w, padding=1)                  # ★ 합성곱 한 줄
    print(f"{name:8s} 출력 shape : {tuple(out.shape)}")
    ax[i].imshow(out[0,0], cmap="gray"); ax[i].set_title(name); ax[i].axis("off")

plt.tight_layout(); plt.show()

> **관찰 포인트 ★**: 커널 숫자만 바꿨는데 **세로선만 남거나, 가로선만 남거나, 흐려집니다.**
> **커널 = 찾고 싶은 무늬**라는 말이 여기서 눈으로 확인됩니다.

> **핵심 메시지 ★★**: 지금은 **우리가 숫자를 정해서** 넣었습니다.
> CNN 은 이 9개 숫자를 **학습으로 스스로 찾습니다.**
> *"어떤 무늬를 찾아야 고양이를 잘 맞히는가"* 를 데이터로부터 배우는 것입니다.
> (1주차에 말한 *"특징을 사람이 설계하지 않는다"* 가 바로 이 이야기입니다.)

> `padding=1` 을 줘서 출력이 **32×32 로 유지**되는 것을 확인하세요. 다음 실습의 공식 그대로입니다.

## 실습 2 — 출력 크기 계산 ★

```
                    입력 − 커널 + 2 × 패딩
        출력 크기 = ────────────────────────  + 1        ※ 나눗셈은 내림
                          스트라이드
```

| 인자 | 뜻 | 감각 |
|---|---|---|
| **`in_channels`** | 입력 채널 수 | 컬러 이미지는 3 (R,G,B) |
| **`out_channels`** | **커널의 개수** ★ | 32 개면 서로 다른 무늬 32 가지를 찾는다 |
| **`kernel_size`** | 창의 크기 | 3×3 이 표준 (VGG 이후) |
| **`stride`** | 몇 칸씩 옮기나 | 2 면 출력이 절반으로 |
| **`padding`** | 가장자리에 0 을 두르는 폭 | 1 이면 3×3 커널에서 **크기가 유지**된다 |

> ### 먼저 손으로 계산하세요. 그다음에 아래 셀을 실행합니다.
> 순서를 바꾸면 아무도 계산하지 않습니다.
> ① `k=3, s=1, p=1` ② `k=3, s=1, p=0` ③ `k=3, s=2, p=1` ④ `k=5, s=1, p=2`

In [ ]:
# 셀 3 — 손으로 계산하고, 코드로 확인한다
x = torch.randn(1, 3, 32, 32)          # 배치1, 채널3, 32×32

설정 = [
    dict(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
    dict(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=0),
    dict(in_channels=3, out_channels=16, kernel_size=3, stride=2, padding=1),
    dict(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2),
]

for cfg in 설정:
    계산 = (32 - cfg["kernel_size"] + 2*cfg["padding"]) // cfg["stride"] + 1
    실제 = nn.Conv2d(**cfg)(x).shape
    print(f"k={cfg['kernel_size']} s={cfg['stride']} p={cfg['padding']} "
          f"| 손계산 {계산:2d} | 실제 {tuple(실제)}")

> **관찰 포인트 ★**: 출력 shape 은 `(1, 16, H, W)` 입니다.
> **채널이 3 → 16 으로 바뀐 것**에 주목하세요. `out_channels=16` 이 커널 16개라는 뜻입니다.

> **핵심 메시지 ★**: `k=3, s=1, p=1` 은 **32 → 32**, `k=5, s=1, p=2` 도 **32 → 32** 입니다.
> 규칙: **`padding = (kernel_size − 1) / 2` 이면 크기가 유지**됩니다. 실무에서 계속 씁니다.

> **함정 ★**: 나눗셈은 **내림**입니다. `stride=2` 에서 홀수가 나오면 잘립니다.
> 2교시 실습 4에서 FC 입력 크기를 계산할 때 여기서 틀리면 오류가 납니다.

In [ ]:
# 셀 4 — 풀링 : 파라미터가 없는 축소 연산
pool = nn.MaxPool2d(2)
y = torch.tensor([[[[1., 3., 2., 0.],
                    [4., 2., 1., 1.],
                    [0., 4., 5., 1.],
                    [1., 1., 2., 3.]]]])
print("입력 4×4 :\n", y[0,0])
print("\nMaxPool2d(2) 결과 2×2 :\n", pool(y)[0,0])
print("\nAvgPool2d(2) 결과 2×2 :\n", nn.AvgPool2d(2)(y)[0,0])
print("\n풀링의 파라미터 수 :", sum(p.numel() for p in pool.parameters()), "  ← 학습하지 않는다")

| | 하는 일 | 왜 |
|---|---|---|
| **MaxPool** | 창에서 최댓값 | *"이 근처에 그 무늬가 있었다"* 만 남긴다. **CNN 의 기본** |
| AvgPool | 창에서 평균 | 부드럽게. 마지막 층에서 가끔 |

| 풀링의 효과 | 설명 |
|---|---|
| 크기를 줄인다 | 계산량·메모리 감소 |
| **위치 변화에 둔감해진다** ★ | 고양이가 조금 움직여도 같은 답 |
| 시야가 넓어진다 | 뒤쪽 층의 3×3 커널이 원본에서는 더 넓은 영역을 본다 |

> **파라미터가 없습니다.** 학습하는 것이 아니라 **그냥 줄이는 연산**입니다.
> 그래서 `Conv → BN → ReLU → Pool` 을 한 묶음으로 쌓습니다 — 2교시의 기본 블록입니다.

## 실습 3 — 특징맵 시각화 ★

> hook 코드는 같은 폴더의 `feature_map_hook.py` 에 완성본이 있습니다.
> 이 실습의 목적은 코딩이 아니라 **보는 것**입니다.

In [ ]:
# 셀 5 — 학습 안 한 CNN 의 특징맵
torch.manual_seed(0)
net = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(),      # 1층
    nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),     # 2층
)
print(net)

feats = {}
def hook(name):
    def fn(m, i, o): feats[name] = o.detach()
    return fn

h1 = net[0].register_forward_hook(hook("1층"))
h2 = net[3].register_forward_hook(hook("2층"))

x = train[7][0].unsqueeze(0)          # (1,3,32,32)
_ = net(x)

for name in ["1층", "2층"]:
    f = feats[name][0]
    print(f"{name} 특징맵 shape : {tuple(f.shape)}")
    fig, ax = plt.subplots(1, 8, figsize=(14, 2))
    for i in range(8):
        ax[i].imshow(f[i], cmap="viridis"); ax[i].axis("off")
    fig.suptitle(f"{name} 특징맵 (앞 8개 채널)"); plt.show()

h1.remove(); h2.remove()              # ★ hook 은 쓰고 나면 떼어 낸다

> **관찰 포인트 ★**: **채널마다 반응하는 곳이 다릅니다.**
> 어떤 채널은 윤곽에, 어떤 채널은 배경에 밝게 반응합니다.
> 아직 **학습 전**인데도 그렇습니다 — 커널이 무작위여도 서로 다른 무늬를 잡기 때문입니다.

> **핵심 메시지 ★★**: **층이 깊어질수록 보는 것이 달라집니다.**
> ```
>   1층  : 선, 색 경계 같은 단순한 것
>   2층  : 그 선들이 모인 모양 (모서리, 질감)
>   더 깊은 층 : 눈, 바퀴 같은 부분  →  결국 물체 전체
> ```
> **사람이 이 단계를 설계하지 않았습니다.** 데이터에서 스스로 나온 것이고,
> 이것이 딥러닝의 정의적 특징(1주차)입니다.

> 2층 특징맵이 **16×16** 인 것을 확인하세요 — 사이의 `MaxPool2d(2)` 때문입니다.

> **막히면**: hook 이 안 걸리면 층 인덱스(`net[0]`, `net[3]`)를 확인하세요.
> `print(net)` 으로 번호를 볼 수 있습니다.

In [ ]:
# 셀 6 (참고) — 배포된 도우미로 같은 일을 더 짧게
from feature_map_hook import FeatureCatcher, show_feature_maps, show_kernels, conv_out_size

with FeatureCatcher(net, {"1층": net[0], "2층": net[3]}) as c:
    net(x)                            # ★ with 를 벗어나면 hook 이 자동으로 떨어진다

show_feature_maps(c.feats["1층"], "1층")
show_kernels(net[0])                  # 커널 자체도 그려 본다 (학습 전이라 잡음처럼 보인다)

print("\n출력 크기 공식 검산 :", conv_out_size(32, kernel=3, stride=2, padding=1))

---

### 이 노트북 체크리스트

- [ ] MLP 가 이미지에 불리한 이유 **두 가지**를 말할 수 있다 ★★
- [ ] 커널을 직접 바꿔 가며 필터 결과가 달라지는 것을 봤다
- [ ] `out_channels` 가 커널 개수라는 것을 안다
- [ ] **출력 크기를 손으로 계산**하고 실제와 대조했다 ★
- [ ] `padding=(k−1)/2` 이면 크기가 유지되는 것을 안다
- [ ] 풀링에 파라미터가 없다는 것과 그 목적을 안다
- [ ] 1층·2층 특징맵을 보고 차이를 설명할 수 있다